# Тематическое моделирование

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Тематическое моделирование».

## Что делает метод

Тематическое моделирование автоматически находит скрытые темы в большой коллекции текстов. Каждая тема — это набор характерных слов, а каждый документ описывается как смесь нескольких тем. Разметка заранее не нужна: метод обучается без учителя.

Это полезно, когда документов слишком много, чтобы прочитать их вручную: тысячи статей, тезисов конференции, ответов в опросе. Тематическая модель даёт обзорную карту: о чём этот массив текстов и в какой пропорции.

В этом блокноте мы применим латентное размещение Дирихле (LDA) — классический метод тематического моделирования. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте библиотекаря, которому принесли гору книг без оглавлений и без каталога. Ему нужно разложить их по тематическим полкам. Он листает страницы и замечает: в одних книгах часто встречаются слова «реактор», «турбина», «мощность» — это, наверное, энергетика. В других — «пациент», «диагноз», «препарат» — это медицина. Некоторые книги охватывают сразу две темы.

Именно так работает тематическое моделирование. Алгоритм не читает и не понимает тексты — он ищет группы слов, которые часто появляются вместе в одних документах. Такие группы и называют «темами». Каждый документ при этом может относиться к нескольким темам одновременно — в разных пропорциях.

Конкретный алгоритм, который мы используем — **LDA** (Latent Dirichlet Allocation, или «Латентное размещение Дирихле»):
- «Латентное» — темы скрыты, их не видно напрямую в данных, они выводятся из статистики слов.
- «Дирихле» — статистическое распределение, описывающее смеси тем.

Для учёного это способ быстро получить «карту» большого текстового массива: что в нём есть, в какой пропорции, и как тематика меняется во времени или между группами документов.

Терминологический минимум для этого блокнота:
- **Корпус** — коллекция всех текстов, которые анализируются.
- **Документ** — один текст в корпусе (статья, аннотация, ответ в анкете).
- **Словарь** — множество всех уникальных слов в корпусе.
- **Мешок слов** (bag-of-words) — представление документа как вектора счётчиков слов. Порядок слов теряется.
- **Матрица «документ–слово»** — таблица, где строки — документы, столбцы — слова, ячейки — количество вхождений.
- **Скрытая тема** — группа слов с высоким совместным весом, выделенная алгоритмом.
- **Состав документа** (topic mixture) — вектор долей тем: какую часть документа занимает каждая тема. Сумма равна 1.
- **CountVectorizer** — инструмент scikit-learn для построения матрицы «документ–слово».
- **Обучение без учителя** — алгоритм не знает правильных ответов, он сам находит структуру в данных.

## 1. Установка библиотек

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сформируем корпус коротких текстов из трёх скрытых тем. Мы знаем темы заранее, чтобы потом проверить, восстановит ли их модель, но самой модели метки не передаются.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# Три скрытые темы заданы наборами характерных слов.
topic_words = {
    "климат": ["температура", "осадки", "атмосфера", "потепление",
               "ледник", "прогноз", "выбросы"],
    "медицина": ["пациент", "терапия", "диагноз", "клетка",
                 "препарат", "иммунитет", "симптом"],
    "энергетика": ["реактор", "турбина", "генератор", "сеть",
                   "нагрузка", "мощность", "топливо"],
}
common = ["исследование", "данные", "анализ", "результат", "модель"]

documents, true_topic = [], []
for _ in range(120):
    topic = rng.choice(list(topic_words))
    words = list(rng.choice(topic_words[topic], size=8)) \
        + list(rng.choice(common, size=4))
    rng.shuffle(words)
    documents.append(" ".join(words))
    true_topic.append(topic)

data = pd.DataFrame({"текст": documents, "истинная_тема": true_topic})
print(f"Документов в корпусе: {len(data)}")
data.head()

## 4. Применение метода

### Шаг 1. Векторизация текстов и обучение LDA

Прежде чем применить LDA, нужно перевести тексты в числовой вид. `CountVectorizer` строит матрицу «документ–слово»: каждая строка — один документ, каждый столбец — одно слово из словаря, значение — сколько раз это слово встречается в документе. Порядок слов при этом теряется — LDA работает именно с такой «корзиной слов».

Параметр `n_topics = 3` задаёт число тем, которые алгоритм ищет. Мы намеренно ставим три — столько же, сколько закладывали при создании корпуса. В реальных задачах число тем подбирают по метрикам когерентности и содержательной осмысленности.

`LatentDirichletAllocation` находит темы, итеративно оптимизируя соответствие между словами и темами. `fit_transform` обучает модель и сразу возвращает матрицу состава документов.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(documents)
vocabulary = np.array(vectorizer.get_feature_names_out())

n_topics = 3
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42,
                                learning_method="batch", max_iter=30)
doc_topic = lda.fit_transform(counts)   # доля каждой темы в документе
print(f"Размер матрицы «документ - тема»: {doc_topic.shape}")
print(f"Размер словаря: {len(vocabulary)} слов")

### Шаг 2. Характерные слова тем

После обучения в `lda.components_` хранятся веса слов для каждой темы — чем выше вес, тем характернее слово для темы. Выведем топ-7 слов каждой найденной темы.

Важно: темы пронумерованы произвольно — «Тема 0» не обязана совпадать с «климатом» только потому, что так записано в данных. Смысл темы определяет исследователь по набору слов.

Сравните выведенные слова с теми, что мы заложили при создании корпуса.

In [ ]:
def top_words(model, vocab, n=7):
    """Возвращает наиболее весомые слова каждой темы."""
    rows = []
    for k, weights in enumerate(model.components_):
        top = vocab[np.argsort(weights)[::-1][:n]]
        rows.append({"тема": f"Тема {k}", "характерные слова": ", ".join(top)})
    return pd.DataFrame(rows)


top_words(lda, vocabulary)

### Шаг 3. Состав документов

Каждый документ описывается вектором долей тем — «состав документа». Сумма долей по всем темам равна 1. Выведем несколько документов и сопоставим с истинной темой, чтобы понять, как хорошо LDA восстановил структуру.

Обратите внимание: документы с истинной темой «климат» должны иметь высокую долю какой-то одной найденной темы — но нумерация может не совпадать.

In [ ]:
mix = pd.DataFrame(doc_topic.round(2),
                   columns=[f"Тема {k}" for k in range(n_topics)])
mix["истинная_тема"] = true_topic
mix.head(8)

### Шаг 4. Визуализация: как хорошо LDA восстановил темы

Для каждой истинной темы усредним вектора состава документов и выведем столбчатую диаграмму. Это покажет, насколько чётко каждая истинная тема соответствует одной найденной.

In [ ]:
import matplotlib.pyplot as plt

# Для каждой истинной темы усредним доли найденных тем.
summary = (pd.DataFrame(doc_topic, columns=[f"Тема {k}" for k in range(n_topics)])
           .assign(истинная=true_topic)
           .groupby("истинная").mean())

fig, ax = plt.subplots(figsize=(9.5, 5.6))
summary.plot(kind="bar", ax=ax, color=VIZ["series"][:n_topics], width=0.75)
ax.set_title("Средний состав документов по истинным темам")
ax.set_xlabel("Истинная тема корпуса")
ax.set_ylabel("Средняя доля найденной темы")
ax.legend(title="Найденная тема", loc="upper right")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()

**Как читать этот график:**

По горизонтальной оси — истинные темы корпуса (которые мы знаем, потому что данные синтетические). По вертикальной оси — средняя доля каждой найденной темы в документах этой истинной темы.

Что значит хорошая модель: каждая группа столбцов (истинная тема) должна иметь один явно доминирующий столбец и маленькие остальные. Это значит, что одна найденная тема соответствует одной истинной.

Что значит плохая модель: столбцы примерно одинаковые — LDA не смог разделить темы. Возможные причины: слишком мало документов, слишком короткие тексты, неправильно выбрано число тем.

В реальных задачах истинных меток нет — интерпретируете график вы сами, как исследователь.

## 5. Интерпретация результата

- **Характерные слова** показывают смысл каждой найденной темы. Если слова темы образуют осмысленную группу — модель уловила реальную закономерность; «слова вперемешку» означают, что тем взято слишком много или мало.
- **Состав документа** — это вектор долей; сумма долей равна единице. Документ обычно принадлежит не одной теме, а смеси, что отражает реальную многоплановость текстов.
- **График**: для качественной модели каждая истинная тема корпуса соответствует преимущественно одной найденной теме — столбцы выражено доминируют по диагонали.
- Число тем — главный параметр. Его подбирают по интерпретируемости и по метрикам связности тем; «правильного» числа из данных автоматически не следует.

Помните: соответствие найденных и истинных тем здесь можно проверить только потому, что данные синтетические. На реальных текстах метки отсутствуют, и темы оценивает эксперт.

## 6. Попробуйте на своих данных

Замените демонстрационный корпус своими текстами.

1. Подготовьте таблицу с колонкой текстов (по одному документу на строку). Желательно заранее убрать стоп-слова и привести слова к начальной форме.
2. Снимите комментарии в ячейке ниже, укажите файл и при необходимости измените число тем `n_topics`.
3. Выполните ячейки раздела 4.

## 7. Поэкспериментируйте сами

Измените один параметр и повторно выполните ячейки раздела 4:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `n_topics = 5` | Запросить 5 тем вместо 3 | Темы начнут дробиться на части или появятся «мусорные» темы |
| `n_topics = 2` | Запросить только 2 темы | Темы сольются — посмотрите, какие два объединятся |
| Длина документа — `size=4` | В `rng.choice(topic_words[topic], size=4)` | Короткие тексты — LDA труднее разделяет темы |
| `max_iter = 5` | Очень мало итераций | Недообученная модель — темы плохо разделены |
| `max_iter = 100` | Больше итераций | Улучшается ли чёткость разделения? |

Дополнительный эксперимент — проверьте, что сумма долей тем для любого документа равна 1:
```python
print(doc_topic[0].sum())  # должно быть 1.0
```
Это фундаментальное свойство LDA: документ описывается распределением вероятностей по темам.

In [ ]:
# --- Шаблон загрузки своих данных ---
# raw = pd.read_csv("my_corpus.csv")
# documents = raw["текст"].astype(str).tolist()
#
# counts = vectorizer.fit_transform(documents)
# vocabulary = np.array(vectorizer.get_feature_names_out())
# n_topics = 5                              # подберите под свой корпус
# lda = LatentDirichletAllocation(n_components=n_topics, random_state=42,
#                                 learning_method="batch", max_iter=30)
# doc_topic = lda.fit_transform(counts)
# print(top_words(lda, vocabulary))

## 8. Краткие выводы

- Тематическое моделирование (LDA) находит скрытые темы в корпусе текстов без разметки, основываясь на совместной встречаемости слов.
- Каждый документ описывается как смесь тем — вектор долей, сумма которых равна 1.
- Число тем — главный параметр; его нельзя определить автоматически, подбирают по когерентности и интерпретируемости.
- Темы не пронумерованы по смыслу: соответствие между «Темой 0» и реальным понятием определяет исследователь по характерным словам.
- Синтетические данные позволили проверить модель — в реальных задачах истинных меток нет.
- Для коротких текстов и более семантически богатых тем лучше работают модели на основе эмбеддингов (BERTopic).
- LDA чувствителен к предобработке: без лемматизации и удаления стоп-слов темы засоряются служебными словами.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Вы запустили LDA с `n_topics = 3` и получили «Тему 0» со словами: «исследование, данные, результат, модель, анализ, прогноз, метод». Можно ли считать эту тему содержательной и использовать её для интерпретации корпуса?
2. На столбчатой диаграмме видно, что для всех трёх истинных тем один и тот же столбец «Тема 1» имеет долю около 0.33. Что это означает с точки зрения качества модели и каким параметром это можно попробовать исправить?
3. Исследователь интерпретировал тему по топ-5 словам, присвоил ей название «Климат и энергетика» и стал использовать состав документов как признаки для дальнейшего анализа. Какую ошибку он мог допустить при выборе числа тем, и как проверить, что число тем выбрано обоснованно?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Нет, тема неинформативна: она состоит из общенаучных слов-«сорняков», которые встречаются во всех документах равномерно. Это признак того, что стоп-слова не были удалены при предобработке. Такая тема не несёт тематического смысла и загрязняет результаты; её следует устранить через предобработку, а не увеличением числа тем.
2. Примерно равные доли всех найденных тем в каждой истинной группе означают, что LDA не смог разделить темы — модель «размазала» состав по всем темам равномерно. Это может быть следствием слишком коротких документов, недостаточного числа итераций (`max_iter`) или неоптимального числа тем. Стоит попробовать увеличить длину документов или количество итераций.
3. При слишком малом числе тем разные реальные темы сливаются в одну — отсюда «климат и энергетика» в одной теме. При слишком большом — темы дробятся на фрагменты. Для обоснованного выбора используют метрику когерентности (Topic Coherence, например, c_v или u_mass): она оценивает, насколько слова темы действительно встречаются вместе в тексте. Сравнивают её значения при разном `n_topics` и выбирают максимум с учётом содержательной интерпретируемости.
</details>
